<div style="background: linear-gradient(135deg, #0d6efd 0%, #6610f2 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🌌 01 — A Geometria das Matrizes</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 01 · Bloco 04 · Transformações</p>
</div>

## 🎯 Objetivos deste notebook

Ao finalizar este notebook você vai:
- Entender que **matrizes são transformações**, não apenas tabelas de números
- Implementar rotação, escala e reflexão em 2D com NumPy
- Visualizar o efeito geométrico de cada transformação
- Compor múltiplas transformações em sequência
- Conectar esses conceitos com Data Augmentation em Visão Computacional

---

## 1️⃣ A Grande Ideia: Matriz = Transformação

Quando você multiplica uma matriz por um vetor, **você não está fazendo conta** — você está **movendo, girando ou deformando** aquele ponto no espaço.

Cada tipo de matriz produz um tipo diferente de transformação:
- **Escala** → estica ou comprime os eixos
- **Rotação** → gira os pontos ao redor da origem
- **Reflexão** → espelha os pontos em relação a um eixo

Vamos ver cada uma com código e gráficos.

## 2️⃣ Matriz de Escala

Uma matriz de escala multiplica cada coordenada por um fator:
```
[sx   0 ]
[ 0  sy ]
```
- `sx > 1`: estica no eixo X
- `sy < 1`: comprime no eixo Y
- `sx = sy`: escala uniforme (zoom)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Criar um quadrado unitário (4 vértices + fechamento)
quadrado = np.array([
    [0, 0], [1, 0], [1, 1], [0, 1], [0, 0]  # fecha o polígono
]).T  # shape (2, 5) → cada coluna é um ponto

# Matriz de Escala: esticar 2x no X, comprimir 0.5x no Y
sx, sy = 2.0, 0.5
M_escala = np.array([[sx, 0],
                     [0, sy]])

# Aplicar transformação
resultado = M_escala @ quadrado

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.plot(quadrado[0], quadrado[1], 'b-o', label='Original', linewidth=2)
ax.plot(resultado[0], resultado[1], 'r-s', label=f'Escala ({sx}x, {sy}x)', linewidth=2)
ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
ax.set_title('Transformação de Escala', fontsize=14, fontweight='bold')
plt.show()

## 3️⃣ Matriz de Rotação

Para girar um ponto `(x, y)` por um ângulo θ ao redor da origem:
```
[cos(θ)  -sin(θ)]
[sin(θ)   cos(θ)]
```

**Propriedade importante:** O determinante desta matriz é sempre 1 (preserva áreas — só gira, não estica).

In [ ]:
def matriz_rotacao(angulo_graus):
    """Cria matriz de rotação 2D a partir de um ângulo em graus."""
    theta = np.radians(angulo_graus)
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

# Rotacionar o quadrado em 45 graus
M_rot = matriz_rotacao(45)
rot_resultado = M_rot @ quadrado

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.plot(quadrado[0], quadrado[1], 'b-o', label='Original', linewidth=2)
ax.plot(rot_resultado[0], rot_resultado[1], 'g-^', label='Rotação 45°', linewidth=2)

# Mostrar ângulo
ax.annotate('45°', xy=(0.5, 0.3), fontsize=14, color='green', fontweight='bold')
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-0.5, 1.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
ax.set_title('Transformação de Rotação', fontsize=14, fontweight='bold')
plt.show()

# Verificar: determinante deve ser 1
print(f'Determinante da matriz de rotação: {np.linalg.det(M_rot):.6f}')

## 4️⃣ Rotações Múltiplas — Animação Conceitual

Vamos girar o mesmo quadrado em vários ângulos para ver o efeito acumulado:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
cores = plt.cm.viridis(np.linspace(0, 1, 8))

for i, angulo in enumerate(range(0, 360, 45)):
    M = matriz_rotacao(angulo)
    resultado = M @ quadrado
    ax.plot(resultado[0], resultado[1], '-', color=cores[i], 
            linewidth=2, label=f'{angulo}°')

ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='upper right')
ax.set_title('Quadrado rotacionado a cada 45°', fontsize=14, fontweight='bold')
plt.show()

## 5️⃣ Matriz de Reflexão

Reflexão espelha os pontos em relação a um eixo:

| Reflexão | Matriz |
|---|---|
| Eixo X (horizontal) | `[[1, 0], [0, -1]]` |
| Eixo Y (vertical)   | `[[-1, 0], [0, 1]]` |
| Origem               | `[[-1, 0], [0, -1]]` |

**Detalhe:** O determinante de uma reflexão é **-1** (inverte a orientação).

In [ ]:
# Reflexão no eixo X (inverte Y)
M_ref_x = np.array([[1,  0],
                     [0, -1]])

# Reflexão no eixo Y (inverte X)
M_ref_y = np.array([[-1, 0],
                     [ 0, 1]])

ref_x = M_ref_x @ quadrado
ref_y = M_ref_y @ quadrado

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, resultado, titulo in zip(axes, [ref_x, ref_y], 
                                  ['Reflexão no Eixo X', 'Reflexão no Eixo Y']):
    ax.plot(quadrado[0], quadrado[1], 'b-o', label='Original', linewidth=2)
    ax.plot(resultado[0], resultado[1], 'r-s', label='Refletido', linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_title(titulo, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Det(Reflexão X) = {np.linalg.det(M_ref_x):.0f} (inverte orientação!)')
print(f'Det(Reflexão Y) = {np.linalg.det(M_ref_y):.0f}')

## 6️⃣ Composição de Transformações

O poder real aparece quando **encadeamos** transformações. A ordem importa!

**Regra:** Primeiro aplica a transformação mais à direita.
- `M_rot @ M_escala @ ponto` → primeiro escala, depois gira
- `M_escala @ M_rot @ ponto` → primeiro gira, depois escala

In [ ]:
# Composição: Rotação 30° + Escala 1.5x
M_composta_1 = matriz_rotacao(30) @ np.array([[1.5, 0], [0, 1.5]])
M_composta_2 = np.array([[1.5, 0], [0, 1.5]]) @ matriz_rotacao(30)

r1 = M_composta_1 @ quadrado
r2 = M_composta_2 @ quadrado

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, res, titulo in zip(axes, [r1, r2],
                            ['Escala → Rotação', 'Rotação → Escala']):
    ax.plot(quadrado[0], quadrado[1], 'b-o', label='Original', linewidth=2)
    ax.plot(res[0], res[1], 'r-s', label='Transformado', linewidth=2)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-1.5, 2.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_title(titulo, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Neste caso específico (escala uniforme), o resultado é o mesmo!
# Mas com escala não-uniforme, a ordem IMPORTA.

## 7️⃣ Conexão com Visão Computacional

Tudo que você acabou de fazer é a base de:
- **Data Augmentation**: rotacionar, espelhar e escalar imagens antes do treino
- **Homografia**: transformar perspectiva (retificação de documentos)
- **Warp Affine no OpenCV**: `cv2.getRotationMatrix2D()` gera exatamente estas matrizes

> 💡 Na Fase 03 você vai usar `cv2.warpAffine()` — que aplica estas matrizes a imagens reais.

## 🏋️ Questões para Praticar

### Q1. Cisalhamento (Shear)
Implemente uma **matriz de cisalhamento** (shear) que desloca o eixo X proporcionalmente ao Y:
```
[1  k]
[0  1]
```
Teste com `k = 0.5` e visualize o quadrado deformado.

### Q2. Sequência de 3 Transformações
Aplique ao quadrado: (1) escala 2x no X, (2) rotação 60°, (3) reflexão no eixo Y. Visualize o resultado final e cada passo intermediário.

### Q3. Determinante e Área
Calcule o determinante da sua matriz composta da Q2. O que ele diz sobre a **área** do quadrado após a transformação?

### Q4. Inversa
Calcule `np.linalg.inv()` da matriz de rotação de 45°. Multiplique a inversa pelo resultado rotacionado. Você recupera o original?

In [ ]:
# ============================================
# Resolva as questões aqui
# ============================================

# Q1: Cisalhamento


# Q2: Sequência de transformações


# Q3: Determinante e área


# Q4: Inversa
